In this script we will develop a basic training pipeline and try to develop the maximum functionality manually with <b>no nn.Modeule</b>

In [1]:
import torch
from sklearn.model_selection import train_test_split

In [2]:
data = torch.rand(10,5)
data

tensor([[0.0777, 0.8383, 0.0058, 0.3128, 0.8871],
        [0.6427, 0.5528, 0.2969, 0.2252, 0.0463],
        [0.8981, 0.4815, 0.2820, 0.0965, 0.5060],
        [0.6925, 0.4746, 0.2626, 0.9445, 0.4314],
        [0.1965, 0.3267, 0.7308, 0.2422, 0.9299],
        [0.7154, 0.0629, 0.3124, 0.7484, 0.2355],
        [0.9843, 0.6596, 0.8659, 0.1168, 0.6966],
        [0.4595, 0.7799, 0.9868, 0.1242, 0.4024],
        [0.3725, 0.7663, 0.1856, 0.4259, 0.5140],
        [0.0953, 0.4617, 0.3362, 0.9935, 0.1275]])

In [3]:
x = data[:,0:-1]
y = data[:,-1]

four features and one target

In [4]:
X_train, X_test, y_train, y_test = train_test_split(x,y,test_size=0.2,random_state=42)

In [5]:
class BasicANN():
    def __init__(self,x):
        # requires_grad True means that we want to calculate the gradients for these parameters during backpropagation
        self.weights = torch.rand(x.shape[1],1,requires_grad=True)
        self.bias = torch.rand(1,requires_grad=True)

    def forward(self,x):
        # forward pass we will calculate the linear transformation of the input data and then apply the sigmoid activation 
        # function to get the predicted output
        # z = activation_function(x*weights + bias)
        z = torch.matmul(x,self.weights) + self.bias
        y_pred = torch.sigmoid(z)
        return y_pred
    
    def loss(self,y_pred,y):
        # to avoid log(0) which is undefined we will clamp the predicted values between a small value (epsilon) and 1-epsilon
        # this will ensure that the predicted values are always between epsilon and 1-epsilon, preventing any issues with log(0)
        # binary cross entropy loss is calculated as -y*log(y_pred) - (1-y)*log(1-y_pred) and then we take the mean of the loss 
        # values to get the average loss over the batch
        epsilon = 1e-7
        y_pred = torch.clamp(y_pred,epsilon,1-epsilon)
        loss = -y*torch.log(y_pred) - (1-y)*torch.log(1-y_pred) # binary cross endtropy
        return loss.mean()

In [6]:
# parameters of the model are the weights and bias which we want to optimize during training
learning_rate = 0.01
epochs = 10

In [7]:
# create an instance of the BasicANN class
model = BasicANN(X_train)

In [9]:
print('Weights:', model.weights)
print('Bias:', model.bias)

Weights: tensor([[0.1174],
        [0.9600],
        [0.8392],
        [0.9920]], requires_grad=True)
Bias: tensor([0.0792], requires_grad=True)


In [10]:
# ------ 1 Epoch -------
# 1. forward propagation
# 2. calculate loss
# 3. backward propagation
# 4. update parameters

In [12]:
# 1. forward propagation
y_pred = model.forward(X_train)
# 2. calculate loss
loss = model.loss(y_pred,y_train)
print('Loss:', loss.item())
# 3. backward propagation
loss.backward()
# 4. update parameters
with torch.no_grad():
    model.weights -= learning_rate * model.weights.grad
    model.bias -= learning_rate * model.bias.grad
    # zero the gradients after updating
    model.weights.grad.zero_()
    model.bias.grad.zero_()

Loss: 0.9082491397857666


In [14]:
for epoch in range(epochs):
    # 1. forward propagation
    y_pred = model.forward(X_train)
    # 2. calculate loss
    loss = model.loss(y_pred,y_train)
    print(f'# Epoch {epoch+1}/{epochs}, Loss: {loss.item():.4f}')
    # 3. backward propagation
    loss.backward()
    # 4. update parameters
    with torch.no_grad(): 
        # we use torch.no_grad() to disable gradient tracking during the parameter update step, 
        # since we don't want to calculate gradients for the parameters during this step
        # weights_new = weights_old - learning_rate * weights_old.grad # d(loss)/d(weights_old)
        model.weights -= learning_rate * model.weights.grad
        model.bias -= learning_rate * model.bias.grad
        # zero the gradients after updating
        model.weights.grad.zero_()
        model.bias.grad.zero_()

# Epoch 1/10, Loss: 0.8914
# Epoch 2/10, Loss: 0.8900
# Epoch 3/10, Loss: 0.8885
# Epoch 4/10, Loss: 0.8871
# Epoch 5/10, Loss: 0.8856
# Epoch 6/10, Loss: 0.8842
# Epoch 7/10, Loss: 0.8827
# Epoch 8/10, Loss: 0.8813
# Epoch 9/10, Loss: 0.8799
# Epoch 10/10, Loss: 0.8785
